# RFM Customer Segmentation - Online Retail

**Goal:** group customers by Recency, Frequency and Monetary value, then suggest a marketing action per segment.

**Dataset:** Online Retail (UCI / Kaggle), UK online retailer, Dec 2010 to Dec 2011, prices in GBP.

Place the data file in the `data/` folder before running.

## 1. Load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# UCI (Excel) version:
df = pd.read_excel("../data/Online Retail.xlsx")
# Kaggle (CSV) version - use this instead:
# df = pd.read_csv("../data/data.csv", encoding="ISO-8859-1")

df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
print("Before cleaning:", df.shape)
df.info()
print(df.isnull().sum())
df.describe()

## 2. Clean data

Remove missing CustomerIDs, cancelled invoices (start with `C`), non-positive quantity or price, and duplicates.

In [ ]:
df = df.dropna(subset=["CustomerID"])
df = df[~df["InvoiceNo"].astype(str).str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["UnitPrice"] > 0)]
df = df.drop_duplicates().copy()
df["CustomerID"] = df["CustomerID"].astype(int)
df["TotalPrice"] = df["Quantity"] * df["UnitPrice"]

print("After cleaning:", df.shape)
print("Unique customers:", df["CustomerID"].nunique())

## 3. Calculate Recency, Frequency, Monetary

Snapshot date = last invoice date + 1 day (not today's date, otherwise everyone looks inactive).

In [ ]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
print("Snapshot date:", snapshot_date)

rfm = df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("TotalPrice", "sum"),
)
rfm.head()

## 4. Score 1-5 and create segments

Recency: fewer days is better, so the most recent customers get 5. `rank(method="first")` avoids duplicate bin edges in `qcut`.

In [ ]:
rfm["R_Score"] = pd.qcut(rfm["Recency"].rank(method="first"), 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm["RFM_Score"] = rfm["R_Score"].astype(str) + rfm["F_Score"].astype(str) + rfm["M_Score"].astype(str)

def assign_segment(row):
    r, f = row["R_Score"], row["F_Score"]
    if r >= 4 and f >= 4: return "Champions"
    if r >= 3 and f >= 3: return "Loyal"
    if r >= 4 and f <= 2: return "Potential Loyalists"
    if r <= 2 and f >= 3: return "At Risk"
    if r <= 2 and f <= 2: return "Lost"
    return "Need Attention"

rfm["Segment"] = rfm.apply(assign_segment, axis=1)
rfm.head()

## 5. Segment summary

In [ ]:
order = ["Champions", "Loyal", "Potential Loyalists", "Need Attention", "At Risk", "Lost"]

summary = rfm.groupby("Segment").agg(
    Customers=("Recency", "size"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean"),
    Total_Revenue=("Monetary", "sum"),
)
summary["Customer_Share_%"] = summary["Customers"] / summary["Customers"].sum() * 100
summary["Revenue_Share_%"] = summary["Total_Revenue"] / summary["Total_Revenue"].sum() * 100
summary = summary.reindex(order).round(1)
summary

## 6. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=summary.index, y=summary["Customers"], ax=ax, color="#4C72B0")
ax.set_title("Number of Customers per Segment")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("../images/1_segment_size.png", dpi=150)
plt.show()

**Insight:** _write one line here after you see the chart._

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(x=summary.index, y=summary["Revenue_Share_%"], ax=ax, color="#55A868")
ax.set_title("Revenue Share by Segment (%)")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.savefig("../images/2_revenue_share.png", dpi=150)
plt.show()

**Insight:** _write one line here._

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(rfm["Recency"], bins=30, ax=axes[0], color="#C44E52")
axes[0].set_title("Recency (days)")
sns.histplot(np.log1p(rfm["Frequency"]), bins=30, ax=axes[1], color="#8172B2")
axes[1].set_title("Frequency (log scale)")
sns.histplot(np.log1p(rfm["Monetary"]), bins=30, ax=axes[2], color="#CCB974")
axes[2].set_title("Monetary (log scale)")
plt.tight_layout()
plt.savefig("../images/3_rfm_distributions.png", dpi=150)
plt.show()

**Insight:** _write one line here._

In [ ]:
avg = summary[["Avg_Recency", "Avg_Frequency", "Avg_Monetary"]]
norm = (avg - avg.min()) / (avg.max() - avg.min())

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(norm, annot=avg, fmt=".1f", cmap="YlGnBu", ax=ax)
ax.set_title("Average R, F, M per Segment")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("../images/4_segment_heatmap.png", dpi=150)
plt.show()

**Insight:** _write one line here._

## 7. Export results

In [ ]:
rfm.to_csv("../reports/rfm_customers.csv")
summary.to_csv("../reports/segment_summary.csv")

## 8. Business recommendations

See `reports/business_recommendations.md`. Summary:

- **Champions:** reward and retain, no heavy discounts.
- **Loyal:** upsell and loyalty tiers.
- **Potential Loyalists:** push a second purchase.
- **Need Attention:** time-limited offers.
- **At Risk:** win-back campaign.
- **Lost:** low-cost last-chance email, then stop.